In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import shap
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import MiniBatchDictionaryLearning

from PyBioMed.PyProtein.CTD import CalculateCTD
from PyBioMed.PyProtein import AAComposition   
from PyBioMed.PyProtein import QuasiSequenceOrder   
from PyBioMed.Pyprotein import CalculateConjointTriad  
from PyBioMed.PyProtein.CTD import CalculateC,CalculateT,CalculateD  

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

c:\Users\WanGXing\anaconda3\envs\my-rdkit-env\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ic50_data_train1 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_train_cv1.csv',usecols=['sequence','ic50'])
ic50_data_test1 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_test_cv1.csv',usecols=['sequence','ic50'])
ic50_data_train2 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_train_cv2.csv',usecols=['sequence','ic50'])
ic50_data_test2 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_test_cv2.csv',usecols=['sequence','ic50'])
ic50_data_train3 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_train_cv3.csv',usecols=['sequence','ic50'])
ic50_data_test3 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_test_cv3.csv',usecols=['sequence','ic50'])
ic50_data_train4 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_train_cv4.csv',usecols=['sequence','ic50'])
ic50_data_test4 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_test_cv4.csv',usecols=['sequence','ic50'])
ic50_data_train5 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_train_cv5.csv',usecols=['sequence','ic50'])
ic50_data_test5 = pd.read_csv('G:\VScode\Working_files\ic50_dataset\ic50_data_test_cv5.csv',usecols=['sequence','ic50'])

noic50_data = pd.read_csv('G:\VScode\Working_files\ic50_dataset\stap_data_distribution.csv',usecols=['sequence','ic50_data'])
noic50_data.drop(noic50_data.index[noic50_data[noic50_data['ic50_data']=='yes'].index], inplace=True)

ic50_data = pd.read_csv('G:\VScode\Working_files\ic50_dataset\stap_data_distribution.csv',usecols=['sequence','ic50_data'])
ic50_data.drop(ic50_data.index[ic50_data[ic50_data['ic50_data']=='no'].index], inplace=True)


In [3]:

def get_feature(data):
    if isinstance(data, pd.DataFrame):
        temp = np.array(data)
    else:
        temp = data
    
    CTD_list = []
    AAC_list = []
    DPC_list = []
    three_mer_list = []
    QSO_list = []
    CT_list = []
    CTDC_list = []
    CTDT_list = []
    CTDD_list = []
    label_list = []
    for i in temp:
        CTD_list.append(list(CalculateCTD(i[0]).values()))
        AAC_list.append(list(AAComposition.CalculateAAComposition(i[0]).values()))
        DPC_list.append(list(AAComposition.CalculateDipeptideComposition(i[0]).values()))
        three_mer_list.append(list(AAComposition.GetSpectrumDict(i[0]).values()))
        QSO_list.append(list(QuasiSequenceOrder.GetQuasiSequenceOrder(i[0], maxlag=30, weight=0.1).values()))
        CT_list.append(list(CalculateConjointTriad(i[0]).values()))
        CTDC_list.append(list(CalculateC(i[0]).values()))
        CTDT_list.append(list(CalculateT(i[0]).values()))
        CTDD_list.append(list(CalculateD(i[0]).values()))
        label_list.append(i[1])
    return CTD_list,AAC_list,DPC_list,three_mer_list,QSO_list,CT_list,CTDC_list,CTDT_list,CTDD_list,label_list



CTD_list_train1,AAC_list_train1,DPC_list_train1,three_mer_list_train1,QSO_list_train1,CT_list_train1,CTDC_list_train1,CTDT_list_train1,CTDD_list_train1,label_list_train1=get_feature(ic50_data_train1)
CTD_list_test1,AAC_list_test1,DPC_list_test1,three_mer_list_test1,QSO_list_test1,CT_list_test1,CTDC_list_test1,CTDT_list_test1,CTDD_list_test1,label_list_test1=get_feature(ic50_data_test1)

CTD_list_train2,AAC_list_train2,DPC_list_train2,three_mer_list_train2,QSO_list_train2,CT_list_train2,CTDC_list_train2,CTDT_list_train2,CTDD_list_train2,label_list_train2=get_feature(ic50_data_train2)
CTD_list_test2,AAC_list_test2,DPC_list_test2,three_mer_list_test2,QSO_list_test2,CT_list_test2,CTDC_list_test2,CTDT_list_test2,CTDD_list_test2,label_list_test2=get_feature(ic50_data_test2)

CTD_list_train3,AAC_list_train3,DPC_list_train3,three_mer_list_train3,QSO_list_train3,CT_list_train3,CTDC_list_train3,CTDT_list_train3,CTDD_list_train3,label_list_train3=get_feature(ic50_data_train3)
CTD_list_test3,AAC_list_test3,DPC_list_test3,three_mer_list_test3,QSO_list_test3,CT_list_test3,CTDC_list_test3,CTDT_list_test3,CTDD_list_test3,label_list_test3=get_feature(ic50_data_test3)

CTD_list_train4,AAC_list_train4,DPC_list_train4,three_mer_list_train4,QSO_list_train4,CT_list_train4,CTDC_list_train4,CTDT_list_train4,CTDD_list_train4,label_list_train4=get_feature(ic50_data_train4)
CTD_list_test4,AAC_list_test4,DPC_list_test4,three_mer_list_test4,QSO_list_test4,CT_list_test4,CTDC_list_test4,CTDT_list_test4,CTDD_list_test4,label_list_test4=get_feature(ic50_data_test4)

CTD_list_train5,AAC_list_train5,DPC_list_train5,three_mer_list_train5,QSO_list_train5,CT_list_train5,CTDC_list_train5,CTDT_list_train5,CTDD_list_train5,label_list_train5=get_feature(ic50_data_train5)
CTD_list_test5,AAC_list_test5,DPC_list_test5,three_mer_list_test5,QSO_list_test5,CT_list_test5,CTDC_list_test5,CTDT_list_test5,CTDD_list_test5,label_list_test5=get_feature(ic50_data_test5)

CTD_list_unlabel,AAC_list_unlabel,DPC_list_unlabel,three_mer_list_unlabel,QSO_list_unlabel,CT_list_unlabel,CTDC_list_unlabel,CTDT_list_unlabel,CTDD_list_unlabel,label_list_unlabel=get_feature(noic50_data)

CTD_list_inlabel,AAC_list_inlabel,DPC_list_inlabel,three_mer_list_inlabel,QSO_list_inlabel,CT_list_inlabel,CTDC_list_inlabel,CTDT_list_inlabel,CTDD_list_inlabel,label_list_inlabel=get_feature(ic50_data)
 

In [4]:

CTD_train1,AAC_train1,DPC_train1,three_mer_train1,QSO_train1,CT_train1,CTDC_train1,CTDT_train1,CTDD_train1,label_train1 = CTD_list_train1,AAC_list_train1,DPC_list_train1,three_mer_list_train1,QSO_list_train1,CT_list_train1,CTDC_list_train1,CTDT_list_train1,CTDD_list_train1,label_list_train1
CTD_train2,AAC_train2,DPC_train2,three_mer_train2,QSO_train2,CT_train2,CTDC_train2,CTDT_train2,CTDD_train2,label_train2 = CTD_list_train2,AAC_list_train2,DPC_list_train2,three_mer_list_train2,QSO_list_train2,CT_list_train2,CTDC_list_train2,CTDT_list_train2,CTDD_list_train2,label_list_train2
CTD_train3,AAC_train3,DPC_train3,three_mer_train3,QSO_train3,CT_train3,CTDC_train3,CTDT_train3,CTDD_train3,label_train3 = CTD_list_train3,AAC_list_train3,DPC_list_train3,three_mer_list_train3,QSO_list_train3,CT_list_train3,CTDC_list_train3,CTDT_list_train3,CTDD_list_train3,label_list_train3
CTD_train4,AAC_train4,DPC_train4,three_mer_train4,QSO_train4,CT_train4,CTDC_train4,CTDT_train4,CTDD_train4,label_train4 = CTD_list_train4,AAC_list_train4,DPC_list_train4,three_mer_list_train4,QSO_list_train4,CT_list_train4,CTDC_list_train4,CTDT_list_train4,CTDD_list_train4,label_list_train4
CTD_train5,AAC_train5,DPC_train5,three_mer_train5,QSO_train5,CT_train5,CTDC_train5,CTDT_train5,CTDD_train5,label_train5 = CTD_list_train5,AAC_list_train5,DPC_list_train5,three_mer_list_train5,QSO_list_train5,CT_list_train5,CTDC_list_train5,CTDT_list_train5,CTDD_list_train5,label_list_train5
# 

In [5]:
def Integration_features():

    all_feature_name = ['CTD', 'AAC', 'DPC', 'three_mer', 'QSO', 'CT', 'CTDC', 'CTDT', 'CTDD']
    all_train1 = [CTD_list_train1, AAC_list_train1, DPC_list_train1, three_mer_list_train1, QSO_list_train1, CT_list_train1, CTDC_list_train1, CTDT_list_train1, CTDD_list_train1]
    all_test1 = [CTD_list_test1, AAC_list_test1, DPC_list_test1, three_mer_list_test1, QSO_list_test1, CT_list_test1, CTDC_list_test1, CTDT_list_test1, CTDD_list_test1]
    all_train2 = [CTD_list_train2, AAC_list_train2, DPC_list_train2, three_mer_list_train2, QSO_list_train2, CT_list_train2, CTDC_list_train2, CTDT_list_train2, CTDD_list_train2]
    all_test2 = [CTD_list_test2, AAC_list_test2, DPC_list_test2, three_mer_list_test2, QSO_list_test2, CT_list_test2, CTDC_list_test2, CTDT_list_test2, CTDD_list_test2]
    all_train3 = [CTD_list_train3, AAC_list_train3, DPC_list_train3, three_mer_list_train3, QSO_list_train3, CT_list_train3, CTDC_list_train3, CTDT_list_train3, CTDD_list_train3]
    all_test3 = [CTD_list_test3, AAC_list_test3, DPC_list_test3, three_mer_list_test3, QSO_list_test3, CT_list_test3, CTDC_list_test3, CTDT_list_test3, CTDD_list_test3]
    all_train4 = [CTD_list_train4, AAC_list_train4, DPC_list_train4, three_mer_list_train4, QSO_list_train4, CT_list_train4, CTDC_list_train4, CTDT_list_train4, CTDD_list_train4]
    all_test4 = [CTD_list_test4, AAC_list_test4, DPC_list_test4, three_mer_list_test4, QSO_list_test4, CT_list_test4, CTDC_list_test4, CTDT_list_test4, CTDD_list_test4]
    all_train5 = [CTD_list_train5, AAC_list_train5, DPC_list_train5, three_mer_list_train5, QSO_list_train5, CT_list_train5, CTDC_list_train5, CTDT_list_train5, CTDD_list_train5]
    all_test5 = [CTD_list_test5, AAC_list_test5, DPC_list_test5, three_mer_list_test5, QSO_list_test5, CT_list_test5, CTDC_list_test5, CTDT_list_test5, CTDD_list_test5]
    # 
    all_unlabel = [CTD_list_unlabel, AAC_list_unlabel, DPC_list_unlabel, three_mer_list_unlabel, QSO_list_unlabel, CT_list_unlabel, CTDC_list_unlabel, CTDT_list_unlabel, CTDD_list_unlabel]

    return all_feature_name, all_train1, all_test1, all_train2, all_test2, all_train3, all_test3, all_train4, all_test4, all_train5, all_test5, all_unlabel

all_feature_name, all_train1, all_test1, all_train2, all_test2, all_train3, all_test3, all_train4, all_test4, all_train5, all_test5, all_unlabel = Integration_features()
# 

In [7]:

def find_common_numbers(arrays):

    count_dict = {}
    

    for array in arrays:

        for num in array:
            if num in count_dict:
                count_dict[num] += 1
            else:
                count_dict[num] = 1
    

    uniportants = [num for num, count in count_dict.items() if count >= 4]
    
    return uniportants


def model_fit(feature_train, label_train, featuer_test, label_test, model_name, picture=True, important=True):
    if model_name == 'RF':
        model = RandomForestRegressor(n_estimators=80, max_depth=15, random_state=4)
    if model_name == 'GB':
        model = GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=3,min_samples_split=2,min_samples_leaf=1,
                                            max_features=None,subsample=1.0,random_state=42,loss='squared_error',verbose=0)
    if model_name == 'DT':
        model = DecisionTreeRegressor(criterion='squared_error',max_depth=None,min_samples_split=2,min_samples_leaf=1,min_weight_fraction_leaf=0.0,
                                        max_features=None,max_leaf_nodes=None,min_impurity_decrease=0.0,random_state=None,ccp_alpha=0.0)
    if model_name == 'Lasso':
        preprocessor = make_pipeline(StandardScaler(with_mean=False))
        model = Lasso(alpha=0.1,max_iter=5000,tol=0.0001)
        model.transform_alpha = model.alpha
    if model_name == 'Ridge':
        preprocessor = make_pipeline(StandardScaler(with_mean=False))
        model = Ridge(alpha=1.0,max_iter=None,tol=0.0001)
        model.transform_alpha = model.alpha
    if model_name == 'ElasticNet':
        preprocessor = make_pipeline(StandardScaler(with_mean=False))
        model = ElasticNet(alpha=1.0,l1_ratio=0.5,max_iter=5000,tol=0.0001)
        model.transform_alpha = model.alpha
    if model_name == 'SVR':
        preprocessor = make_pipeline(StandardScaler(with_mean=False))
        model = SVR(C=1.0,kernel='rbf',degree=3,gamma='auto',tol=0.001)
    if model_name == 'LR':
        model = LinearRegression(fit_intercept=True,copy_X=True,n_jobs=None)
    if model_name == 'Knn':
        preprocessor = make_pipeline(StandardScaler(with_mean=False))
        model = KNeighborsRegressor(n_neighbors=5,weights='uniform',algorithm='auto',leaf_size=30,p=2,metric='minkowski',
                                        metric_params=None,n_jobs=None)

    model.fit(feature_train, label_train)

    if model_name == 'Lasso' or model_name == 'Ridge' or model_name == 'ElasticNet' or model_name == 'SVR' or model_name == 'Knn':

        full_pipeline = make_pipeline(preprocessor, model)

        full_pipeline.fit(feature_train, label_train)

    y_pred = model.predict(featuer_test)


    mse = mean_squared_error(label_test, y_pred)
    r2 = r2_score(label_test, y_pred)
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"R^2 Score: {r2:.2f}")

    if picture:

        y_test_temp = np.array(label_test)
        plt.scatter(y_test_temp, y_pred)
        plt.xlabel("True IC50 values")
        plt.ylabel("Predicted IC50 values")
        plt.title("True vs Predicted IC50 values")
        plt.plot([min(y_test_temp), max(y_test_temp)], [min(y_test_temp), max(y_test_temp)], color='red', linestyle='--')
        plt.show()
    
    unimportant_features = []
    if important:

        if model_name == 'RF' or model_name == 'GB':
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(feature_train)
        elif model_name == 'DT':
            explainer = shap.Explainer(model, feature_perturbation='interventional', check_additivity=False)
            shap_values = explainer.shap_values(feature_train)
        elif model_name == 'Lasso':
            explainer = shap.Explainer(full_pipeline.named_steps['lasso'], feature_train.iloc[[0]])
            shap_values = explainer(feature_train)
        elif model_name == 'Ridge':
            explainer = shap.Explainer(full_pipeline.named_steps['ridge'], feature_train.iloc[[0]])
            shap_values = explainer(feature_train)
        elif model_name == 'ElasticNet':
            explainer = shap.Explainer(full_pipeline.named_steps['elasticnet'], feature_train.iloc[[0]])
            shap_values = explainer(feature_train)
        elif model_name == 'LR':
            explainer = shap.LinearExplainer(model, masker=feature_train)
            shap_values = explainer.shap_values(feature_train)
        elif model_name == 'SVR':
            explainer = shap.Explainer(full_pipeline.named_steps['svr'].predict, feature_train.iloc[[0]])
            shap_values = explainer(feature_train)
        elif model_name == 'Knn':
            explainer = shap.Explainer(model.predict, feature_train.iloc[[0]])
            shap_values = explainer(feature_train)


        if model_name == 'RF' or model_name == 'GB' or model_name == 'DT' or model_name == 'LR':
            mean_abs_shap = np.abs(shap_values).mean(axis=0)
        if model_name == 'Lasso' or model_name == 'Ridge' or model_name == 'ElasticNet' or model_name == 'SVR' or model_name == 'Knn':
            mean_abs_shap = np.abs(shap_values.values).mean(axis=0)


        threshold = mean_abs_shap.mean() 


        unimportant_features = [feature for feature, shap_val in zip(feature_train.columns, mean_abs_shap) if shap_val < threshold]




    return mse, r2, unimportant_features, model

def model_fit_5cv(feature_train1, label_train1, feature_test1, label_test1, feature_train2, label_train2, feature_test2, label_test2, feature_train3, label_train3, feature_test3, 
                  label_test3, feature_train4, label_train4, feature_test4, label_test4, feature_train5, label_train5, feature_test5, label_test5, model_name):
    feature_train1 = pd.DataFrame(feature_train1)
    feature_train2 = pd.DataFrame(feature_train2)
    feature_train3 = pd.DataFrame(feature_train3)
    feature_train4 = pd.DataFrame(feature_train4)
    feature_train5 = pd.DataFrame(feature_train5)
    feature_test1 = pd.DataFrame(feature_test1)
    feature_test2 = pd.DataFrame(feature_test2)
    feature_test3 = pd.DataFrame(feature_test3)
    feature_test4 = pd.DataFrame(feature_test4)
    feature_test5 = pd.DataFrame(feature_test5)
    
    mse1, r21, unimportant_features1, _ = model_fit(feature_train1, label_train1, feature_test1, label_test1, model_name, picture=False, important=True)
    mse2, r22, unimportant_features2, _ = model_fit(feature_train2, label_train2, feature_test2, label_test2, model_name, picture=False, important=True)
    mse3, r23, unimportant_features3, _ = model_fit(feature_train3, label_train3, feature_test3, label_test3, model_name, picture=False, important=True)
    mse4, r24, unimportant_features4, _ = model_fit(feature_train4, label_train4, feature_test4, label_test4, model_name, picture=False, important=True)
    mse5, r25, unimportant_features5, _ = model_fit(feature_train5, label_train5, feature_test5, label_test5, model_name, picture=False, important=True)
    
    mse = (mse1 + mse2 + mse3 + mse4 + mse5)/5
    r2 = (r21 + r22 + r23 + r24 + r25)/5


    array = [unimportant_features1, unimportant_features2, unimportant_features3, unimportant_features4, unimportant_features5]
    uniportants = find_common_numbers(array)
    return  mse, r2, uniportants

def concat_fea(data1, data2):
    if isinstance(data1, pd.DataFrame):
        temp1 = np.array(data1)
    else:
        temp1 = data1
    if isinstance(data2, pd.DataFrame):
        temp2 = np.array(data2)
    else:
        temp2 = data2
    
    concat_temp = []
    k1 = 0
    for fea1 in temp1:
        k2 = 0
        for fea2 in temp2:
            if k1 == k2:
                concat = np.concatenate((fea1, fea2), axis=0)
                concat_temp.append(concat)
            k2+=1
        k1+=1
    return concat_temp


def get_name_index(name):
    name_index = [index for index, feature_name in enumerate(all_feature_name) if feature_name == name]
    name_index = name_index[0]
    return name_index

In [8]:

def find_max_two(*args):


    max1, max2 = float('-inf'), float('-inf')
    index_max1, index_max2 = -1, -1


    for i, num in enumerate(args):
        if num > max1:
            max2 = max1
            index_max2 = index_max1
            max1 = num
            index_max1 = i
        elif num > max2:
            max2 = num
            index_max2 = i


    above_threshold_count = sum([1 for val in (max1, max2) if val > 0.3])


    return index_max1, index_max2, above_threshold_count



def baseline_model(model):
    mse_CTD, r2_CTD, unimportants_CTD = model_fit_5cv(CTD_list_train1, label_list_train1, CTD_list_test1, label_list_test1, CTD_list_train2, label_list_train2, CTD_list_test2, label_list_test2, 
                                                      CTD_list_train3, label_list_train3, CTD_list_test3, label_list_test3, CTD_list_train4, label_list_train4, CTD_list_test4, label_list_test4, 
                                                      CTD_list_train5, label_list_train5, CTD_list_test5, label_list_test5, model)
    mse_AAC, r2_AAC, unimportants_AAC = model_fit_5cv(AAC_list_train1, label_list_train1, AAC_list_test1, label_list_test1, AAC_list_train2, label_list_train2, AAC_list_test2, label_list_test2, 
                                                      AAC_list_train3, label_list_train3, AAC_list_test3, label_list_test3, AAC_list_train4, label_list_train4, AAC_list_test4, label_list_test4, 
                                                      AAC_list_train5, label_list_train5, AAC_list_test5, label_list_test5, model)
    mse_DPC, r2_DPC, unimportants_DPC = model_fit_5cv(DPC_list_train1, label_list_train1, DPC_list_test1, label_list_test1, DPC_list_train2, label_list_train2, DPC_list_test2, label_list_test2, 
                                                      DPC_list_train3, label_list_train3, DPC_list_test3, label_list_test3, DPC_list_train4, label_list_train4, DPC_list_test4, label_list_test4, 
                                                      DPC_list_train5, label_list_train5, DPC_list_test5, label_list_test5, model)
    mse_three_mer, r2_three_mer, unimportants_three_mer = model_fit_5cv(three_mer_list_train1, label_list_train1, three_mer_list_test1, label_list_test1, three_mer_list_train2, label_list_train2, three_mer_list_test2, label_list_test2, 
                                                                         three_mer_list_train3, label_list_train3, three_mer_list_test3, label_list_test3, three_mer_list_train4, label_list_train4, three_mer_list_test4, label_list_test4, 
                                                                         three_mer_list_train5, label_list_train5, three_mer_list_test5, label_list_test5, model)
    mse_QSO, r2_QSO, unimportants_QSO = model_fit_5cv(QSO_list_train1, label_list_train1, QSO_list_test1, label_list_test1, QSO_list_train2, label_list_train2, QSO_list_test2, label_list_test2, 
                                                      QSO_list_train3, label_list_train3, QSO_list_test3, label_list_test3, QSO_list_train4, label_list_train4, QSO_list_test4, label_list_test4, 
                                                      QSO_list_train5, label_list_train5, QSO_list_test5, label_list_test5, model)
    mse_CT, r2_CT, unimportants_CT = model_fit_5cv(CT_list_train1, label_list_train1, CT_list_test1, label_list_test1, CT_list_train2, label_list_train2, CT_list_test2, label_list_test2, 
                                                      CT_list_train3, label_list_train3, CT_list_test3, label_list_test3, CT_list_train4, label_list_train4, CT_list_test4, label_list_test4, 
                                                      CT_list_train5, label_list_train5, CT_list_test5, label_list_test5, model)
    mse_CTDC, r2_CTDC, unimportants_CTDC = model_fit_5cv(CTDC_list_train1, label_list_train1, CTDC_list_test1, label_list_test1, CTDC_list_train2, label_list_train2, CTDC_list_test2, label_list_test2, 
                                                      CTDC_list_train3, label_list_train3, CTDC_list_test3, label_list_test3, CTDC_list_train4, label_list_train4, CTDC_list_test4, label_list_test4, 
                                                      CTDC_list_train5, label_list_train5, CTDC_list_test5, label_list_test5, model)
    mse_CTDT, r2_CTDT, unimportants_CTDT = model_fit_5cv(CTDT_list_train1, label_list_train1, CTDT_list_test1, label_list_test1, CTDT_list_train2, label_list_train2, CTDT_list_test2, label_list_test2, 
                                                      CTDT_list_train3, label_list_train3, CTDT_list_test3, label_list_test3, CTDT_list_train4, label_list_train4, CTDT_list_test4, label_list_test4, 
                                                      CTDT_list_train5, label_list_train5, CTDT_list_test5, label_list_test5, model)
    mse_CTDD, r2_CTDD, unimportants_CTDD = model_fit_5cv(CTDD_list_train1, label_list_train1, CTDD_list_test1, label_list_test1, CTDD_list_train2, label_list_train2, CTDD_list_test2, label_list_test2, 
                                                      CTDD_list_train3, label_list_train3, CTDD_list_test3, label_list_test3, CTDD_list_train4, label_list_train4, CTDD_list_test4, label_list_test4, 
                                                      CTDD_list_train5, label_list_train5, CTDD_list_test5, label_list_test5, model)
    

    all_name = ['CTD', 'AAC', 'DPC', 'three_mer', 'QSO', 'CT', 'CTDC', 'CTDT', 'CTDD']
    all_r2 = [r2_CTD, r2_AAC, r2_DPC, r2_three_mer, r2_QSO, r2_CT, r2_CTDC, r2_CTDT, r2_CTDD]
    all_mse = [mse_CTD, mse_AAC, mse_DPC, mse_three_mer, mse_QSO, mse_CT, mse_CTDC, mse_CTDT, mse_CTDD]
    all_unimportants = [unimportants_CTD, unimportants_AAC, unimportants_DPC, unimportants_three_mer, unimportants_QSO, unimportants_CT, unimportants_CTDC, unimportants_CTDT, unimportants_CTDD]


    index_max1, index_max2, above_threshold_count = find_max_two(*all_r2)

    return above_threshold_count, all_unimportants[index_max1], all_name[index_max1], all_unimportants[index_max2], all_name[index_max2]


In [9]:

def concat_fea(data1, data2):
    if isinstance(data1, pd.DataFrame):
        temp1 = np.array(data1)
    else:
        temp1 = data1
    if isinstance(data2, pd.DataFrame):
        temp2 = np.array(data2)
    else:
        temp2 = data2
    
    concat_temp = []
    k1 = 0
    for fea1 in temp1:
        k2 = 0
        for fea2 in temp2:
            if k1 == k2:
                concat = np.concatenate((fea1, fea2), axis=0)
                concat_temp.append(concat)
            k2+=1
        k1+=1
    return concat_temp

def concat_features(count, feature1, feature2, unimportant_features1, unimportant_features2):
    feature1 = pd.DataFrame(feature1).drop(unimportant_features1, axis=1)
    feature2 = pd.DataFrame(feature2).drop(unimportant_features2, axis=1)
    if count == 2:
        feature_concat = concat_fea(feature1, feature2)
    if count == 1:
        feature_concat = feature1
    if count == 0:
        feature_concat = 'None'
    return feature_concat


def get_concat_features(count, feature_name1, feature_name2, unimportant_features1, unimportant_features2):
    concat_train1 = concat_features(count, all_train1[get_name_index(feature_name1)], all_train1[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_train2 = concat_features(count, all_train2[get_name_index(feature_name1)], all_train2[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_train3 = concat_features(count, all_train3[get_name_index(feature_name1)], all_train3[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_train4 = concat_features(count, all_train4[get_name_index(feature_name1)], all_train4[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_train5 = concat_features(count, all_train5[get_name_index(feature_name1)], all_train5[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_test1 = concat_features(count, all_test1[get_name_index(feature_name1)], all_test1[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_test2 = concat_features(count, all_test2[get_name_index(feature_name1)], all_test2[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_test3 = concat_features(count, all_test3[get_name_index(feature_name1)], all_test3[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_test4 = concat_features(count, all_test4[get_name_index(feature_name1)], all_test4[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    concat_test5 = concat_features(count, all_test5[get_name_index(feature_name1)], all_test5[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    return concat_train1, concat_train2, concat_train3, concat_train4, concat_train5, concat_test1, concat_test2, concat_test3, concat_test4, concat_test5


def get_unlabel_concat(count, feature_name1, feature_name2, unimportant_features1, unimportant_features2):
    concat_unlabel = concat_features(count, all_unlabel[get_name_index(feature_name1)], all_unlabel[get_name_index(feature_name2)], unimportant_features1, unimportant_features2)
    return concat_unlabel


def MBDL(X_train, X_test, unlabel):
    dictionary_learner = MiniBatchDictionaryLearning(n_components=2, batch_size=16, alpha=0.2, n_iter=3000, random_state=42)
    X_train_low_rank = dictionary_learner.fit_transform(X_train)
    X_test_low_rank = dictionary_learner.transform(X_test)
    unlabel_low_rank = dictionary_learner.transform(unlabel)
    return X_train_low_rank, X_test_low_rank, unlabel_low_rank


def compare_five_numbers(a, b, c, d, e):
    max_value = a
    max_index = 0
    
    numbers = [b, c, d, e]
    for i, num in enumerate(numbers, start=1):
        if num > max_value:
            max_value = num
            max_index = i
    
    max = max_index + 1 if max_index != 0 else 1
    return max, max_value


def concat_model_5cv(feature_train1, label_train1, feature_test1, label_test1, feature_train2, label_train2, feature_test2, label_test2, feature_train3, label_train3, feature_test3, 
                  label_test3, feature_train4, label_train4, feature_test4, label_test4, feature_train5, label_train5,feature_test5, label_test5, feature_unlabel, model_name):
    
    feature_train1_low_rank, feature_test1_low_rank, unlabel_low_rank1 = MBDL(feature_train1, feature_test1, feature_unlabel)
    feature_train2_low_rank, feature_test2_low_rank, unlabel_low_rank2 = MBDL(feature_train2, feature_test2, feature_unlabel)
    feature_train3_low_rank, feature_test3_low_rank, unlabel_low_rank3 = MBDL(feature_train3, feature_test3, feature_unlabel)
    feature_train4_low_rank, feature_test4_low_rank, unlabel_low_rank4 = MBDL(feature_train4, feature_test4, feature_unlabel)
    feature_train5_low_rank, feature_test5_low_rank, unlabel_low_rank5 = MBDL(feature_train5, feature_test5, feature_unlabel)
    
    mse1, r21, _, model1 = model_fit(feature_train1_low_rank, label_train1, feature_test1_low_rank, label_test1, model_name, picture=False, important=False)
    mse2, r22, _, model2 = model_fit(feature_train2_low_rank, label_train2, feature_test2_low_rank, label_test2, model_name, picture=False, important=False)
    mse3, r23, _, model3 = model_fit(feature_train3_low_rank, label_train3, feature_test3_low_rank, label_test3, model_name, picture=False, important=False)
    mse4, r24, _, model4 = model_fit(feature_train4_low_rank, label_train4, feature_test4_low_rank, label_test4, model_name, picture=False, important=False)
    mse5, r25, _, model5 = model_fit(feature_train5_low_rank, label_train5, feature_test5_low_rank, label_test5, model_name, picture=False, important=False)

    max_index, max_value = compare_five_numbers(r21, r22, r23, r24, r25)
    print('------------------------------------------')
    print('model name: ', model_name)
    print('max r2: ', max_value)
    print('mean r2: ', (r21 + r22 + r23 + r24 + r25)/5)
    print('------------------------------------------')
    if max_index == 1:
        return model1, unlabel_low_rank1, max_value
    if max_index == 2:
        return model2, unlabel_low_rank2, max_value
    if max_index == 3:
        return model3, unlabel_low_rank3, max_value
    if max_index == 4:
        return model4, unlabel_low_rank4, max_value
    if max_index == 5:
        return model5, unlabel_low_rank5, max_value

def calculate_mean_and_std(v1, v2, v3, v4, v5, v6, v7, v8, v9):

    vectors = [v for v in (v1, v2, v3, v4, v5, v6, v7, v8, v9) if v is not None]


    vector_length = len(vectors[0])


    vectors_array = np.array(vectors)

    mean = np.mean(vectors_array, axis=0)

    std = np.std(vectors_array, axis=0)

    return mean, std


def get_bottom_values(a, b, n):

    bottom_indices = np.argsort(b)[:n]
    
    bottom_a = a[bottom_indices]
    bottom_CTD = [CTD_list_unlabel[i] for i in bottom_indices]
    bottom_AAC = [AAC_list_unlabel[i] for i in bottom_indices]
    bottom_DPC = [DPC_list_unlabel[i] for i in bottom_indices]
    bottom_three_mer = [three_mer_list_unlabel[i] for i in bottom_indices]
    bottom_QSO = [QSO_list_unlabel[i] for i in bottom_indices]
    bottom_CT = [CT_list_unlabel[i] for i in bottom_indices]
    bottom_CTDC = [CTDC_list_unlabel[i] for i in bottom_indices]
    bottom_CTDT = [CTDT_list_unlabel[i] for i in bottom_indices]
    bottom_CTDD = [CTDD_list_unlabel[i] for i in bottom_indices]
    
    return bottom_a, bottom_CTD, bottom_AAC, bottom_DPC, bottom_three_mer, bottom_QSO, bottom_CT, bottom_CTDC, bottom_CTDT, bottom_CTDD

def IC50_pred(n):
    count_RF, unimportants_RF1, feature_name_RF1, unimportants_RF2, feature_name_RF2 = baseline_model('RF')
    count_GB, unimportants_GB1, feature_name_GB1, unimportants_GB2, feature_name_GB2 = baseline_model('GB')
    count_DT, unimportants_DT1, feature_name_DT1, unimportants_DT2, feature_name_DT2 = baseline_model('DT')
    count_Lasso, unimportants_Lasso1, feature_name_Lasso1, unimportants_Lasso2, feature_name_Lasso2 = baseline_model('Lasso')
    count_Ridge, unimportants_Ridge1, feature_name_Ridge1, unimportants_Ridge2, feature_name_Ridge2 = baseline_model('Ridge')
    count_ElasticNet, unimportants_ElasticNet1, feature_name_ElasticNet1, unimportants_ElasticNet2, feature_name_ElasticNet2 = baseline_model('ElasticNet')
    count_SVR, unimportants_SVR1, feature_name_SVR1, unimportants_SVR2, feature_name_SVR2 = baseline_model('SVR')
    count_LR, unimportants_LR1, feature_name_LR1, unimportants_LR2, feature_name_LR2 = baseline_model('LR')
    count_Knn, unimportants_Knn1, feature_name_Knn1, unimportants_Knn2, feature_name_Knn2 = baseline_model('Knn')

    concat_train1_RF, concat_train2_RF, concat_train3_RF, concat_train4_RF, concat_train5_RF, concat_test1_RF, concat_test2_RF, concat_test3_RF, concat_test4_RF, concat_test5_RF = get_concat_features(count_RF, feature_name_RF1, feature_name_RF2, unimportants_RF1, unimportants_RF2)
    concat_train1_GB, concat_train2_GB, concat_train3_GB, concat_train4_GB, concat_train5_GB, concat_test1_GB, concat_test2_GB, concat_test3_GB, concat_test4_GB, concat_test5_GB = get_concat_features(count_GB, feature_name_GB1, feature_name_GB2, unimportants_GB1, unimportants_GB2)
    concat_train1_DT, concat_train2_DT, concat_train3_DT, concat_train4_DT, concat_train5_DT, concat_test1_DT, concat_test2_DT, concat_test3_DT, concat_test4_DT, concat_test5_DT = get_concat_features(count_DT, feature_name_DT1, feature_name_DT2, unimportants_DT1, unimportants_DT2)
    concat_train1_Lasso, concat_train2_Lasso, concat_train3_Lasso, concat_train4_Lasso, concat_train5_Lasso, concat_test1_Lasso, concat_test2_Lasso, concat_test3_Lasso, concat_test4_Lasso, concat_test5_Lasso = get_concat_features(count_Lasso, feature_name_Lasso1, feature_name_Lasso2, unimportants_Lasso1, unimportants_Lasso2)
    concat_train1_Ridge, concat_train2_Ridge, concat_train3_Ridge, concat_train4_Ridge, concat_train5_Ridge, concat_test1_Ridge, concat_test2_Ridge, concat_test3_Ridge, concat_test4_Ridge, concat_test5_Ridge = get_concat_features(count_Ridge, feature_name_Ridge1, feature_name_Ridge2, unimportants_Ridge1, unimportants_Ridge2)
    concat_train1_ElasticNet, concat_train2_ElasticNet, concat_train3_ElasticNet, concat_train4_ElasticNet, concat_train5_ElasticNet, concat_test1_ElasticNet, concat_test2_ElasticNet, concat_test3_ElasticNet, concat_test4_ElasticNet, concat_test5_ElasticNet = get_concat_features(count_ElasticNet, feature_name_ElasticNet1, feature_name_ElasticNet2, unimportants_ElasticNet1, unimportants_ElasticNet2)
    concat_train1_SVR, concat_train2_SVR, concat_train3_SVR, concat_train4_SVR, concat_train5_SVR, concat_test1_SVR, concat_test2_SVR, concat_test3_SVR, concat_test4_SVR, concat_test5_SVR = get_concat_features(count_SVR, feature_name_SVR1, feature_name_SVR2, unimportants_SVR1, unimportants_SVR2)
    concat_train1_LR, concat_train2_LR, concat_train3_LR, concat_train4_LR, concat_train5_LR, concat_test1_LR, concat_test2_LR, concat_test3_LR, concat_test4_LR, concat_test5_LR = get_concat_features(count_LR, feature_name_LR1, feature_name_LR2, unimportants_LR1, unimportants_LR2)
    concat_train1_Knn, concat_train2_Knn, concat_train3_Knn, concat_train4_Knn, concat_train5_Knn, concat_test1_Knn, concat_test2_Knn, concat_test3_Knn, concat_test4_Knn, concat_test5_Knn = get_concat_features(count_Knn, feature_name_Knn1, feature_name_Knn2, unimportants_Knn1, unimportants_Knn2)

    aa = [concat_train1_RF, concat_train1_GB, concat_train1_DT, concat_train1_Lasso, concat_train1_Ridge, concat_train1_ElasticNet, concat_train1_SVR, concat_train1_LR, concat_train1_Knn]
    none_count = sum(1 for item in aa if item is None)
    if none_count < 3:
        print('There are too many None in the concat_train1_xxx')
    
    pseudo_RF = None
    pseudo_GB = None
    pseudo_DT = None
    pseudo_Lasso = None
    pseudo_Ridge = None
    pseudo_ElasticNet = None
    pseudo_SVR = None
    pseudo_LR = None
    pseudo_Knn = None

    if isinstance(concat_train1_RF, pd.DataFrame) or isinstance(concat_train1_RF, list):
        unlabel_concat_RF = get_unlabel_concat(count_RF, feature_name_RF1, feature_name_RF2, unimportants_RF1, unimportants_RF2)
        model_RF, unlabel_low_RF, max_r2_RF = concat_model_5cv(concat_train1_RF, label_list_train1, concat_test1_RF, label_list_test1, concat_train2_RF, label_list_train2, concat_test2_RF, label_list_test2, concat_train3_RF, label_list_train3, concat_test3_RF, label_list_test3, concat_train4_RF, label_list_train4, concat_test4_RF, label_list_test4, concat_train5_RF, label_list_train5, concat_test5_RF, label_list_test5, unlabel_concat_RF, 'RF')
        if max_r2_RF > 0.3:
            pseudo_RF = model_RF.predict(unlabel_low_RF)
    if isinstance(concat_train1_GB, pd.DataFrame) or isinstance(concat_train1_GB, list):
        unlabel_concat_GB = get_unlabel_concat(count_GB, feature_name_GB1, feature_name_GB2, unimportants_GB1, unimportants_GB2)
        model_GB, unlabel_low_GB, max_r2_GB = concat_model_5cv(concat_train1_GB, label_list_train1, concat_test1_GB, label_list_test1, concat_train2_GB, label_list_train2, concat_test2_GB, label_list_test2, concat_train3_GB, label_list_train3, concat_test3_GB, label_list_test3, concat_train4_GB, label_list_train4, concat_test4_GB, label_list_test4, concat_train5_GB, label_list_train5, concat_test5_GB, label_list_test5, unlabel_concat_GB, 'GB')
        if max_r2_GB > 0.3:
            pseudo_GB = model_GB.predict(unlabel_low_GB)
    if isinstance(concat_train1_DT, pd.DataFrame) or isinstance(concat_train1_DT, list):
        unlabel_concat_DT = get_unlabel_concat(count_DT, feature_name_DT1, feature_name_DT2, unimportants_DT1, unimportants_DT2)
        model_DT, unlabel_low_DT, max_r2_DT = concat_model_5cv(concat_train1_DT, label_list_train1, concat_test1_DT, label_list_test1, concat_train2_DT, label_list_train2, concat_test2_DT, label_list_test2, concat_train3_DT, label_list_train3, concat_test3_DT, label_list_test3, concat_train4_DT, label_list_train4, concat_test4_DT, label_list_test4, concat_train5_DT, label_list_train5, concat_test5_DT, label_list_test5, unlabel_concat_DT, 'DT')
        if max_r2_DT > 0.3:
            pseudo_DT = model_DT.predict(unlabel_low_DT)
    if isinstance(concat_train1_Lasso, pd.DataFrame) or isinstance(concat_train1_Lasso, list):
        unlabel_concat_Lasso = get_unlabel_concat(count_Lasso, feature_name_Lasso1, feature_name_Lasso2, unimportants_Lasso1, unimportants_Lasso2)
        model_Lasso, unlabel_low_Lasso, max_r2_Lasso = concat_model_5cv(concat_train1_Lasso, label_list_train1, concat_test1_Lasso, label_list_test1, concat_train2_Lasso, label_list_train2, concat_test2_Lasso, label_list_test2, concat_train3_Lasso, label_list_train3, concat_test3_Lasso, label_list_test3, concat_train4_Lasso, label_list_train4, concat_test4_Lasso, label_list_test4, concat_train5_Lasso, label_list_train5, concat_test5_Lasso, label_list_test5, unlabel_concat_Lasso, 'Lasso')
        if max_r2_Lasso > 0.3:
            pseudo_Lasso = model_Lasso.predict(unlabel_low_Lasso)
    if isinstance(concat_train1_Ridge, pd.DataFrame) or isinstance(concat_train1_Ridge, list):
        unlabel_concat_Ridge = get_unlabel_concat(count_Ridge, feature_name_Ridge1, feature_name_Ridge2, unimportants_Ridge1, unimportants_Ridge2)
        model_Ridge, unlabel_low_Ridge, max_r2_Ridge = concat_model_5cv(concat_train1_Ridge, label_list_train1, concat_test1_Ridge, label_list_test1, concat_train2_Ridge, label_list_train2, concat_test2_Ridge, label_list_test2, concat_train3_Ridge, label_list_train3, concat_test3_Ridge, label_list_test3, concat_train4_Ridge, label_list_train4, concat_test4_Ridge, label_list_test4, concat_train5_Ridge, label_list_train5, concat_test5_Ridge, label_list_test5, unlabel_concat_Ridge, 'Ridge')
        if max_r2_Ridge > 0.3:
            pseudo_Ridge = model_Ridge.predict(unlabel_low_Ridge)
    if isinstance(concat_train1_ElasticNet, pd.DataFrame) or isinstance(concat_train1_ElasticNet, list):
        unlabel_concat_ElasticNet = get_unlabel_concat(count_ElasticNet, feature_name_ElasticNet1, feature_name_ElasticNet2, unimportants_ElasticNet1, unimportants_ElasticNet2)
        model_ElasticNet, unlabel_low_ElasticNet, max_r2_ElasticNet = concat_model_5cv(concat_train1_ElasticNet, label_list_train1, concat_test1_ElasticNet, label_list_test1, concat_train2_ElasticNet, label_list_train2, concat_test2_ElasticNet, label_list_test2, concat_train3_ElasticNet, label_list_train3, concat_test3_ElasticNet, label_list_test3, concat_train4_ElasticNet, label_list_train4, concat_test4_ElasticNet, label_list_test4, concat_train5_ElasticNet, label_list_train5, concat_test5_ElasticNet, label_list_test5, unlabel_concat_ElasticNet, 'ElasticNet')
        if max_r2_ElasticNet > 0.3:
            pseudo_ElasticNet = model_ElasticNet.predict(unlabel_low_ElasticNet)
    if isinstance(concat_train1_SVR, pd.DataFrame) or isinstance(concat_train1_SVR, list):
        unlabel_concat_SVR = get_unlabel_concat(count_SVR, feature_name_SVR1, feature_name_SVR2, unimportants_SVR1, unimportants_SVR2)
        model_SVR, unlabel_low_SVR, max_r2_SVR = concat_model_5cv(concat_train1_SVR, label_list_train1, concat_test1_SVR, label_list_test1, concat_train2_SVR, label_list_train2, concat_test2_SVR, label_list_test2, concat_train3_SVR, label_list_train3, concat_test3_SVR, label_list_test3, concat_train4_SVR, label_list_train4, concat_test4_SVR, label_list_test4, concat_train5_SVR, label_list_train5, concat_test5_SVR, label_list_test5, unlabel_concat_SVR, 'SVR')
        if max_r2_SVR > 0.3:
            pseudo_SVR = model_SVR.predict(unlabel_low_SVR)
    if isinstance(concat_train1_LR, pd.DataFrame) or isinstance(concat_train1_LR, list):
        unlabel_concat_LR = get_unlabel_concat(count_LR, feature_name_LR1, feature_name_LR2, unimportants_LR1, unimportants_LR2)
        model_LR, unlabel_low_LR, max_r2_LR = concat_model_5cv(concat_train1_LR, label_list_train1, concat_test1_LR, label_list_test1, concat_train2_LR, label_list_train2, concat_test2_LR, label_list_test2, concat_train3_LR, label_list_train3, concat_test3_LR, label_list_test3, concat_train4_LR, label_list_train4, concat_test4_LR, label_list_test4, concat_train5_LR, label_list_train5, concat_test5_LR, label_list_test5, unlabel_concat_LR, 'LR')
        if max_r2_LR > 0.3:
            pseudo_LR = model_LR.predict(unlabel_low_LR)
    if isinstance(concat_train1_Knn, pd.DataFrame) or isinstance(concat_train1_Knn, list):
        unlabel_concat_Knn = get_unlabel_concat(count_Knn, feature_name_Knn1, feature_name_Knn2, unimportants_Knn1, unimportants_Knn2)
        model_Knn, unlabel_low_Knn, max_r2_Knn = concat_model_5cv(concat_train1_Knn, label_list_train1, concat_test1_Knn, label_list_test1, concat_train2_Knn, label_list_train2, concat_test2_Knn, label_list_test2, concat_train3_Knn, label_list_train3, concat_test3_Knn, label_list_test3, concat_train4_Knn, label_list_train4, concat_test4_Knn, label_list_test4, concat_train5_Knn, label_list_train5, concat_test5_Knn, label_list_test5, unlabel_concat_Knn, 'Knn')
        if max_r2_Knn > 0.3:
            pseudo_Knn = model_Knn.predict(unlabel_low_Knn)


    mean, std = calculate_mean_and_std(pseudo_RF, pseudo_GB, pseudo_DT, pseudo_Lasso, pseudo_Ridge, pseudo_ElasticNet, pseudo_SVR, pseudo_LR, pseudo_Knn)

    pseudo_label, pseudo_CTD, pseudo_AAC, pseudo_DPC, pseudo_three_mer, pseudo_QSO, pseudo_CT, pseudo_CTDC, pseudo_CTDT, pseudo_CTDD = get_bottom_values(mean, std, n)

    return pseudo_label, pseudo_CTD, pseudo_AAC, pseudo_DPC, pseudo_three_mer, pseudo_QSO, pseudo_CT, pseudo_CTDC, pseudo_CTDT, pseudo_CTDD, mean
    


In [ ]:

pseudo_label, pseudo_CTD, pseudo_AAC, pseudo_DPC, pseudo_three_mer, pseudo_QSO, pseudo_CT, pseudo_CTDC, pseudo_CTDT, pseudo_CTDD, all_pseudo_label1 = IC50_pred(50)



In [11]:

def add_pseudo_label(pseudo_label):
    add_label1 = list(pseudo_label)
    add_label1.extend(label_train1)
    add_label2 = list(pseudo_label)
    add_label2.extend(label_train2)
    add_label3 = list(pseudo_label)
    add_label3.extend(label_train3)
    add_label4 = list(pseudo_label)
    add_label4.extend(label_train4)
    add_label5 = list(pseudo_label)
    add_label5.extend(label_train5)
    return add_label1, add_label2, add_label3, add_label4, add_label5
def add_pseudo_CTD(pseudo_CTD):
    add_CTD_train1 = list(pseudo_CTD)
    add_CTD_train1.extend(CTD_train1)
    add_CTD_train2 = list(pseudo_CTD)
    add_CTD_train2.extend(CTD_train2)
    add_CTD_train3 = list(pseudo_CTD)
    add_CTD_train3.extend(CTD_train3)
    add_CTD_train4 = list(pseudo_CTD)
    add_CTD_train4.extend(CTD_train4)
    add_CTD_train5 = list(pseudo_CTD)
    add_CTD_train5.extend(CTD_train5)
    return add_CTD_train1, add_CTD_train2, add_CTD_train3, add_CTD_train4, add_CTD_train5
def add_pseudo_AAC(pseudo_AAC):
    add_AAC_train1 = list(pseudo_AAC)
    add_AAC_train1.extend(AAC_train1)
    add_AAC_train2 = list(pseudo_AAC)
    add_AAC_train2.extend(AAC_train2)
    add_AAC_train3 = list(pseudo_AAC)
    add_AAC_train3.extend(AAC_train3)
    add_AAC_train4 = list(pseudo_AAC)
    add_AAC_train4.extend(AAC_train4)
    add_AAC_train5 = list(pseudo_AAC)
    add_AAC_train5.extend(AAC_train5)
    return add_AAC_train1, add_AAC_train2, add_AAC_train3, add_AAC_train4, add_AAC_train5
def add_pseudo_DPC(pseudo_DPC):
    add_DPC_train1 = list(pseudo_DPC)
    add_DPC_train1.extend(DPC_train1)
    add_DPC_train2 = list(pseudo_DPC)
    add_DPC_train2.extend(DPC_train2)
    add_DPC_train3 = list(pseudo_DPC)
    add_DPC_train3.extend(DPC_train3)
    add_DPC_train4 = list(pseudo_DPC)
    add_DPC_train4.extend(DPC_train4)
    add_DPC_train5 = list(pseudo_DPC)
    add_DPC_train5.extend(DPC_train5)
    return add_DPC_train1, add_DPC_train2, add_DPC_train3, add_DPC_train4, add_DPC_train5
def add_pseudo_three_mer(pseudo_three_mer):
    add_three_mer_train1 = list(pseudo_three_mer)
    add_three_mer_train1.extend(three_mer_train1)
    add_three_mer_train2 = list(pseudo_three_mer)
    add_three_mer_train2.extend(three_mer_train2)
    add_three_mer_train3 = list(pseudo_three_mer)
    add_three_mer_train3.extend(three_mer_train3)
    add_three_mer_train4 = list(pseudo_three_mer)
    add_three_mer_train4.extend(three_mer_train4)
    add_three_mer_train5 = list(pseudo_three_mer)
    add_three_mer_train5.extend(three_mer_train5)
    return add_three_mer_train1, add_three_mer_train2, add_three_mer_train3, add_three_mer_train4, add_three_mer_train5
def add_pseudo_QSO(pseudo_QSO):
    add_QSO_train1 = list(pseudo_QSO)
    add_QSO_train1.extend(QSO_train1)
    add_QSO_train2 = list(pseudo_QSO)
    add_QSO_train2.extend(QSO_train2)
    add_QSO_train3 = list(pseudo_QSO)
    add_QSO_train3.extend(QSO_train3)
    add_QSO_train4 = list(pseudo_QSO)
    add_QSO_train4.extend(QSO_train4)
    add_QSO_train5 = list(pseudo_QSO)
    add_QSO_train5.extend(QSO_train5)
    return add_QSO_train1, add_QSO_train2, add_QSO_train3, add_QSO_train4, add_QSO_train5
def add_pseudo_CT(pseudo_CT):
    add_CT_train1 = list(pseudo_CT)
    add_CT_train1.extend(CT_train1)
    add_CT_train2 = list(pseudo_CT)
    add_CT_train2.extend(CT_train2)
    add_CT_train3 = list(pseudo_CT)
    add_CT_train3.extend(CT_train3)
    add_CT_train4 = list(pseudo_CT)
    add_CT_train4.extend(CT_train4)
    add_CT_train5 = list(pseudo_CT)
    add_CT_train5.extend(CT_train5)
    return add_CT_train1, add_CT_train2, add_CT_train3, add_CT_train4, add_CT_train5
def add_pseudo_CTDC(pseudo_CTDC):
    add_CTDC_train1 = list(pseudo_CTDC)
    add_CTDC_train1.extend(CTDC_train1)
    add_CTDC_train2 = list(pseudo_CTDC)
    add_CTDC_train2.extend(CTDC_train2)
    add_CTDC_train3 = list(pseudo_CTDC)
    add_CTDC_train3.extend(CTDC_train3)
    add_CTDC_train4 = list(pseudo_CTDC)
    add_CTDC_train4.extend(CTDC_train4)
    add_CTDC_train5 = list(pseudo_CTDC)
    add_CTDC_train5.extend(CTDC_train5)
    return add_CTDC_train1, add_CTDC_train2, add_CTDC_train3, add_CTDC_train4, add_CTDC_train5
def add_pseudo_CTDT(pseudo_CTDT):
    add_CTDT_train1 = list(pseudo_CTDT)
    add_CTDT_train1.extend(CTDT_train1)
    add_CTDT_train2 = list(pseudo_CTDT)
    add_CTDT_train2.extend(CTDT_train2)
    add_CTDT_train3 = list(pseudo_CTDT)
    add_CTDT_train3.extend(CTDT_train3)
    add_CTDT_train4 = list(pseudo_CTDT)
    add_CTDT_train4.extend(CTDT_train4)
    add_CTDT_train5 = list(pseudo_CTDT)
    add_CTDT_train5.extend(CTDT_train5)
    return add_CTDT_train1, add_CTDT_train2, add_CTDT_train3, add_CTDT_train4, add_CTDT_train5
def add_pseudo_CTDD(pseudo_CTDD):
    add_CTDD_train1 = list(pseudo_CTDD)
    add_CTDD_train1.extend(CTDD_train1)
    add_CTDD_train2 = list(pseudo_CTDD)
    add_CTDD_train2.extend(CTDD_train2)
    add_CTDD_train3 = list(pseudo_CTDD)
    add_CTDD_train3.extend(CTDD_train3)
    add_CTDD_train4 = list(pseudo_CTDD)
    add_CTDD_train4.extend(CTDD_train4)
    add_CTDD_train5 = list(pseudo_CTDD)
    add_CTDD_train5.extend(CTDD_train5)
    return add_CTDD_train1, add_CTDD_train2, add_CTDD_train3, add_CTDD_train4, add_CTDD_train5

In [12]:

label_list_train1, label_list_train2, label_list_train3, label_list_train4, label_list_train5 = add_pseudo_label(pseudo_label)
CTD_list_train1, CTD_list_train2, CTD_list_train3, CTD_list_train4, CTD_list_train5 = add_pseudo_CTD(pseudo_CTD)
AAC_list_train1, AAC_list_train2, AAC_list_train3, AAC_list_train4, AAC_list_train5 = add_pseudo_AAC(pseudo_AAC)
DPC_list_train1, DPC_list_train2, DPC_list_train3, DPC_list_train4, DPC_list_train5 = add_pseudo_DPC(pseudo_DPC)
three_mer_list_train1, three_mer_list_train2, three_mer_list_train3, three_mer_list_train4, three_mer_list_train5 = add_pseudo_three_mer(pseudo_three_mer)
QSO_list_train1, QSO_list_train2, QSO_list_train3, QSO_list_train4, QSO_list_train5 = add_pseudo_QSO(pseudo_QSO)
CT_list_train1, CT_list_train2, CT_list_train3, CT_list_train4, CT_list_train5 = add_pseudo_CT(pseudo_CT)
CTDC_list_train1, CTDC_list_train2, CTDC_list_train3, CTDC_list_train4, CTDC_list_train5 = add_pseudo_CTDC(pseudo_CTDC)
CTDT_list_train1, CTDT_list_train2, CTDT_list_train3, CTDT_list_train4, CTDT_list_train5 = add_pseudo_CTDT(pseudo_CTDT)
CTDD_list_train1, CTDD_list_train2, CTDD_list_train3, CTDD_list_train4, CTDD_list_train5 = add_pseudo_CTDD(pseudo_CTDD)

all_feature_name, all_train1, all_test1, all_train2, all_test2, all_train3, all_test3, all_train4, all_test4, all_train5, all_test5, all_unlabel = Integration_features()

In [ ]:

pseudo_label, pseudo_CTD, pseudo_AAC, pseudo_DPC, pseudo_three_mer, pseudo_QSO, pseudo_CT, pseudo_CTDC, pseudo_CTDT, pseudo_CTDD, all_pseudo_label2 = IC50_pred(50)

